<a href="https://colab.research.google.com/github/abuzar01440/Python-Crash-Course--Google/blob/main/web_scaping_of_profiles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
SEDS CS Faculty Scraper — Nazarbayev University
================================================
Scrapes: https://seds.nu.edu.kz/cs_faculty
Collects per faculty member:
  • Name, Title, Email, Phone, Profile URL, Research Interests

Usage:
    pip install requests beautifulsoup4
    python cs_faculty_scraper.py

Output: cs_faculty.csv (created in the same directory)

NOTE: Run this from your local machine.
      The university website blocks cloud/server IP ranges.
"""

import csv
import re
import time
import sys

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    sys.exit(
        "Missing dependencies.\n"
        "Install them with:  pip install requests beautifulsoup4\n"
    )

# ── Config ──────────────────────────────────────────────────────────────────

FACULTY_PAGE  = "https://seds.nu.edu.kz/cs_faculty"
OUTPUT_FILE   = "cs_faculty.csv"
REQUEST_DELAY = 1.0   # seconds between profile requests (be polite)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

# ── Helpers ──────────────────────────────────────────────────────────────────

def fetch(url: str) -> BeautifulSoup | None:
    """GET a page; return soup or None on failure."""
    try:
        r = SESSION.get(url, timeout=20)
        r.raise_for_status()
        return BeautifulSoup(r.text, "html.parser")
    except requests.RequestException as exc:
        print(f"  [WARN] Could not fetch {url}: {exc}")
        return None


def clean(text: str) -> str:
    """Collapse whitespace and strip."""
    return re.sub(r"\s+", " ", text).strip()

# ── Step 1 — Scrape the main faculty listing ─────────────────────────────────

def scrape_listing() -> list[dict]:
    print(f"[1/2] Fetching faculty listing …  {FACULTY_PAGE}")
    soup = fetch(FACULTY_PAGE)
    if soup is None:
        sys.exit(
            "\nCould not load the faculty page.\n"
            "Make sure you are running this script on your own machine,\n"
            "not on a cloud/VPS server (the site blocks those IPs).\n"
        )

    records: list[dict] = []
    seen_urls: set[str] = set()

    # Every faculty card contains an <a> that links to research.nu.edu.kz
    for a in soup.find_all("a", href=re.compile(r"research\.nu\.edu\.kz/en/persons/")):
        url = a["href"].strip().rstrip("/")
        if url in seen_urls:
            continue
        seen_urls.add(url)

        name = clean(a.get_text())
        if not name:
            continue

        # Walk up the DOM tree to find the card container
        container = a.find_parent(["li", "div"])
        title = email = phone = ""

        if container:
            raw = container.get_text(" ", strip=True)

            # Email
            m = re.search(r"[\w.\-+]+@[\w.\-]+\.[a-z]{2,}", raw, re.I)
            email = m.group() if m else ""

            # Phone (Kazakh format: +7 XXXX XX XX XX or similar)
            m = re.search(r"\+?\s*7[\s\d\(\)\-]{9,17}", raw)
            if m:
                phone = clean(m.group())

            # Title — text block immediately after the name, not a phone/email
            all_texts = [t.strip() for t in container.stripped_strings]
            try:
                idx = next(i for i, t in enumerate(all_texts) if name in t)
                for candidate in all_texts[idx + 1 :]:
                    if (
                        candidate
                        and not re.match(r"[\d\+\(]", candidate)
                        and "@" not in candidate
                        and len(candidate) < 80
                    ):
                        title = candidate
                        break
            except StopIteration:
                pass

        records.append(
            {
                "name": name,
                "title": title,
                "email": email,
                "phone": phone,
                "profile_url": url,
                "research_interests": "",
            }
        )
        print(f"   • {name}  ({title})")

    print(f"\n   → {len(records)} faculty members found.\n")
    return records

# ── Step 2 — Scrape each individual profile ───────────────────────────────────

def scrape_profile(url: str) -> str:
    """
    Visit a research.nu.edu.kz/en/persons/ page and extract research interests.

    Priority order:
      1. <h3>Research interests</h3> section  (explicit field, best quality)
      2. 'PhD projects' <dt>/<dd> pair        (stated PhD supervision topics)
      3. Professional profile text             (sentences mentioning interests)
      4. Fingerprint keywords                  (topic tags as fallback)
    """
    soup = fetch(url)
    if soup is None:
        return ""

    parts: list[str] = []

    # ── 1. "Research interests" heading ──────────────────────────────────────
    for h in soup.find_all(["h2", "h3", "h4"]):
        if "research interest" in h.get_text(strip=True).lower():
            # Collect all text until the next heading
            buf = []
            for sibling in h.next_siblings:
                if sibling.name and sibling.name in ("h2", "h3", "h4"):
                    break
                text = clean(sibling.get_text()) if hasattr(sibling, "get_text") else str(sibling).strip()
                if text:
                    buf.append(text)
            if buf:
                parts.append(" ".join(buf))

    # ── 2. PhD projects <dt>/<dd> ─────────────────────────────────────────────
    for dt in soup.find_all("dt"):
        if "phd project" in clean(dt.get_text()).lower():
            dd = dt.find_next_sibling("dd")
            if dd:
                txt = clean(dd.get_text(" "))
                if txt:
                    parts.append(txt)

    # ── 3. Sentences about research in the personal profile ───────────────────
    if not parts:
        for h in soup.find_all(["h2", "h3"]):
            if "personal profile" in h.get_text(strip=True).lower():
                # gather siblings until next h2/h3
                block_text = []
                for sib in h.next_siblings:
                    if sib.name in ("h2", "h3"):
                        break
                    if hasattr(sib, "get_text"):
                        block_text.append(sib.get_text(" ", strip=True))
                full = " ".join(block_text)
                for sentence in re.split(r"(?<=[.!?])\s+", full):
                    if re.search(r"research interest|research area|working on|focus", sentence, re.I):
                        parts.append(clean(sentence))

    # ── 4. Fingerprint keywords (last resort) ─────────────────────────────────
    if not parts:
        keywords = []
        # fingerprint list items often have a percentage sibling
        for li in soup.select("ul.list-descriptions li, .fingerprint-list li"):
            spans = li.find_all(["span", "a"])
            for s in spans:
                txt = clean(s.get_text())
                if txt and not re.fullmatch(r"\d+%?", txt):
                    keywords.append(txt)
        if keywords:
            parts.append("; ".join(dict.fromkeys(keywords)))

    # Deduplicate and join
    combined = "; ".join(dict.fromkeys(p for p in parts if p))
    return combined

# ── Main ─────────────────────────────────────────────────────────────────────

def main() -> None:
    faculty = scrape_listing()

    print("[2/2] Scraping individual profiles …")
    total = len(faculty)
    for i, member in enumerate(faculty, 1):
        print(f"  [{i:>2}/{total}] {member['name']}")
        member["research_interests"] = scrape_profile(member["profile_url"])
        preview = member["research_interests"]
        if preview:
            print(f"         ↳ {preview[:100]}{'…' if len(preview) > 100 else ''}")
        else:
            print("         ↳ (no research interests found on profile page)")
        time.sleep(REQUEST_DELAY)

    # Write CSV
    fields = ["name", "title", "email", "phone", "profile_url", "research_interests"]
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(faculty)

    print(f"\n✅  Saved {len(faculty)} records → '{OUTPUT_FILE}'")


if __name__ == "__main__":
    main()

[1/2] Fetching faculty listing …  https://seds.nu.edu.kz/cs_faculty
   • Adnan Yazici  ()
   • Benjamin Tyler  ()
   • Antonio Cerone  ()
   • Michael Lewis  ()
   • Hans de Nivelle  ()
   • Siamac Fazli  ()
   • Dimitrios Zormpas  ()
   • Lopez Berengueres Jose Oriol  ()
   • Hashim Ali  ()
   • Anh Tu Nguyen  ()
   • tu.nguyen@nu.edu.kz  ()
   • Askar Boranbayev  ()
   • Min-Ho Lee  ()
   • Meiram Murzabulatov  ()
   • Jurn Gyu Park  ()
   • Sain Saginbekov  ()
   • Zohaib Latif  ()
   • Shinnazar Seytnazarov  ()
   • Hari Mohan Rai  ()
   • Ghani Anwar  ()
   • Talgar Bayan  ()
   • Marat Isteleyev  ()
   • Syed Muhammad Umair Arif  ()
   • Asset Berdibek  ()
   • Irina Dolzhikova  ()
   • Yesdaulet Izenov  ()
   • Adai Shomanov  ()
   • Talgat Manglayev  ()
   • Iliyas Tursynbek  ()

   → 29 faculty members found.

[2/2] Scraping individual profiles …
  [ 1/29] Adnan Yazici
         ↳ (no research interests found on profile page)
  [ 2/29] Benjamin Tyler
         ↳ Languages for pr